


*   Fix detect_ordinal_order_cols. It does take into account case of the letter.
*   List item


*   Using dfs = [train, test] is error prone because everytime we change train, test we need to redefine dfs.
*   List item





BEFORE SAVING VERSION IN KAGGLE

*   SPEED_UP = FALSE
*   make sure to add previous LB to input and check the path



TODO:

*   List item




*   Add a change of ordered cats to numbers
*   Change x, y in arguments of plot functions to x_col, y_col
*   Change final ensemble fit with building models
*   Unocomment parametrs in models















In [ ]:
!pip install --upgrade scikit-learn
!pip install catboost

In [ ]:
# IDEAS TO INCLUDE


# class FeatureEngineer(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self

#     def transform(self, X, y=None):
#         X = pd.DataFrame(X).copy()

#         expressions = [
#             "Humidity_Wind_Index       = Humidity / (Wind_Speed_kmh + 1)",
#             "Temp_Humidity_Interaction = Temperature_C * Humidity",
#             "Evapotranspiration_Proxy  = (Temperature_C * Wind_Speed_kmh * Sunlight_Hours) / 1000",
#             "Soil_Health_Score         = (Organic_Carbon * Soil_pH) / (Electrical_Conductivity + 1e-6)",
#             "Irrigation_Load           = Previous_Irrigation_mm / (Field_Area_hectare + 1e-6)",
#             "Organic_Carbon_per_Area   = Organic_Carbon / (Field_Area_hectare + 1e-6)",
#             "Rain_per_Temp             = Rainfall_mm / (Temperature_C + 1)",
#             "Rain_Sunlight_Ratio       = Rainfall_mm / (Sunlight_Hours + 1)",
#             "Water_Deficit             = Previous_Irrigation_mm - Rainfall_mm",
#             "Soil_Moisture_Stress      = Water_Deficit / (Field_Area_hectare + 1)",
#             "Crop_Stage_Water_Need     = Crop_Growth_Stage * Water_Deficit",
#             "VPD_Proxy                 = (Temperature_C ** 2) * (1 - (Humidity / 100))",
#             "Thermal_Energy_Load       = Temperature_C * Sunlight_Hours",
#             "Moisture_Retention_Index  = Soil_Moisture / (Evapotranspiration_Proxy + 1e-6)",
#             #"Rain_Density              = Rainfall_mm / (Field_Area_hectare + 1e-6)",
#             "Hydro_Chemical_Balance    = (Soil_pH * Soil_Moisture) / (Electrical_Conductivity + 1e-6)",
#             "Irrigation_Urgency        = 1 / (Soil_Moisture * Rainfall_mm + 1e-6)",
#             #"Cooling_Potential         = Soil_Moisture - Evapotranspiration_Proxy"
#         ]

#         for expr in expressions:
#             X.eval(expr, inplace=True)

#         X.drop(columns=['Water_Deficit'], errors='ignore', inplace=True)

#         return X
# def preprocessor_pipeline():
#     return Pipeline([
#         ('encoder',     Encoder()),
#         ('feature_eng', FeatureEngineer()),
#         #('scaler',      Scaler()),
#     ])
# pre = preprocessor_pipeline()
# pre.fit(X_raw_train, y_raw_train)
# #X_pro = pre.transform(X_raw_train)
# X_test_pro = pre.transform(raw_test)

######################################################

# def reduce_mem_usage(dataframe, dataset):
#     print('Reducing memory usage for:', dataset)
#     initial_mem_usage = dataframe.memory_usage().sum() / 1024**2

#     for col in dataframe.columns:
#         col_type = dataframe[col].dtype

#         c_min = dataframe[col].min()
#         c_max = dataframe[col].max()
#         if str(col_type)[:3] == 'int':
#             if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
#                 dataframe[col] = dataframe[col].astype(np.int8)
#             elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
#                 dataframe[col] = dataframe[col].astype(np.int16)
#             elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
#                 dataframe[col] = dataframe[col].astype(np.int32)
#             elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
#                 dataframe[col] = dataframe[col].astype(np.int64)
#         else:
#             if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
#                 dataframe[col] = dataframe[col].astype(np.float16)
#             elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
#                 dataframe[col] = dataframe[col].astype(np.float32)
#             else:
#                 dataframe[col] = dataframe[col].astype(np.float64)

#     final_mem_usage = dataframe.memory_usage().sum() / 1024**2
#     print('--- Memory usage before: {:.2f} MB'.format(initial_mem_usage))
#     print('--- Memory usage after: {:.2f} MB'.format(final_mem_usage))
#     print('--- Decreased memory usage by {:.1f}%\n'.format(100 * (initial_mem_usage - final_mem_usage) / initial_mem_usage))

#     return dataframe





In [ ]:
# The notebook uses variety of estimators depending on the problem. For example completely different models have to be imported for classification and regression problems.
# To save resourses I decided to import functions only as they needed. The commented imports below mean that it may or maynot be imported later
# For proper operation of the notebook be sure to preinstall even commented imports

#----------- base libs --------------
import os
import pickle
import time
import numpy as np
import pandas as pd
import random
import tempfile
import polars as pl
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from joblib import Parallel, delayed, dump, load


#----Plotting-----
import matplotlib.pyplot as plt
import seaborn as sns

#----Preproccessing----
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone

#----Linear Models----
# from sklearn.linear_model import LinearRegression


#----Treemodels----
# from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBRegressor

#----Randomized Search-----
# from sklearn.model_selection import RandomizedSearchCV, KFold, StratifiedKFold, cross_val_score

#----Rand Distributions----
from scipy.stats import randint, uniform, loguniform

SPEED_UP = True # Cut the dataset to speed up. Use for debugging.
DO_IDA = False # Do data explaration. Turn off for trianing to save resources.

TARGET = ["Irrigation_Need"]
ID_COL = ["id"]
SCORING = 'balanced_accuracy' # implemented so far: 'roc_auc'
PREDICTION_AIM = 'category' # implemented so far: 'probability'
MAX_RUNNING_TIME = 43200 # Running time limit on kaggle in seconds.

EXPRESSIONS = [
    "soil_lt_25                = Soil_Moisture < 25",
    "temp_gt_30                = Temperature_C > 30",
   # "Humidity_Wind_Index       = Humidity / (Wind_Speed_kmh + 1.0)",
    "rain_lt_300               = Rainfall_mm < 300",
   # "Temp_Humidity_Interaction = Temperature_C * Humidity",
    "wind_gt_10                = Wind_Speed_kmh > 10",
] # Feature enginering
def is_it_kaggle_notebook():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return True
    elif 'COLAB_RELEASE_TAG' in os.environ:
        return False
    else:
        print('The automatic configuration setup fail. Set it manually. Make sure all the paths are correct.')
        raise

IN_KAGGLE = is_it_kaggle_notebook()

if IN_KAGGLE:

    DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e4")
    LEADERBOARD_STORAGE_DIR = Path('/kaggle/working/Models')
    SEED = None
    N_STEPS = 10

    # Load previous models from previous notebookoutput
    import shutil
    PREVIOUS_OUTPUT_DIR = Path('/kaggle/input/notebooks/hkhk2prod/predict-customer-churn')
    WORKING_DIR = Path('/kaggle/working/')
    if os.path.exists(PREVIOUS_OUTPUT_DIR):
        shutil.copytree(PREVIOUS_OUTPUT_DIR, WORKING_DIR, dirs_exist_ok=True)
        print(f"Successfully copied to: {WORKING_DIR}")
    else:
        print("Source folder not found. Check your path!")
    import warnings
    # Suppress the specific DeprecationWarning from the jupyter_client library
    warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")
    warnings.filterwarnings("ignore", category=FutureWarning, message=".*'n_jobs' has no effect.*")

else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Data/playground-series-s6e4")
    LEADERBOARD_STORAGE_DIR = DATA_DIR / 'Models'
    SEED = None
    N_STEPS = 2

TRAIN_CSV   = "train.csv"
TEST_CSV    = "test.csv"
SAMPLE_CSV     = "sample_submission.csv"
SUBMISSION_FILE_NAME     = 'submission'




CAT_CUTOFF = 5
# Maximal number of unique values in a column for it to be considered categorical.
# It only matters for graph plotting. We make it to avoid column with only 5 unique values to be plotted as a continuous function.




#This list defines the order of cat variables. Whenever you need to order smth just add a list to this. EVERYTHING SHOULD BE LOWER CASE.
ORDER_LISTS = [["poor", "average", "good"],
                  ["low", "medium", "high"],
                  ["easy", "moderate", "hard"],
                  ["no", "yes"]
]
CAT_ORDER_LIST = sum(ORDER_LISTS, start=[])


def cat_order(x):

    """Order on categorical variables: first try to order by CAT_ORDER_LIST, then by lens.
    This orderding only affects the display of categories on the graphs.
    """
    if x.lower() in CAT_ORDER_LIST:
        return CAT_ORDER_LIST.index(x.lower())
    else:
        return len(CAT_ORDER_LIST) + len(x)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
train_df = pd.read_csv(DATA_DIR / TRAIN_CSV )
test_df = pd.read_csv(DATA_DIR / TEST_CSV)
sample_df = pd.read_csv(DATA_DIR / SAMPLE_CSV)
dfs = [train_df, test_df]

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

def get_columns(df=None):
    if df is None:
        df = train_df
    return train_df.drop(columns=ID_COL).columns

def get_predictor_names(df=None):
    if df is None:
        df = train_df
    return train_df.drop(columns=TARGET + ID_COL).columns

def is_num_check(col, for_graph = False, df=None):
    if df is None:
        df = train_df
    return pd.api.types.is_numeric_dtype(df[col]) and (not for_graph or df[col].nunique() > CAT_CUTOFF)


def detect_ordinal_order_cols(df, order_lists=ORDER_LISTS):
    """
    Generates a dict with keys being columns that contain ordered categonical variables, and values are lists of unique values in natural order.
    """
    result = {}

    for col in df.columns:
        vals = list(pd.unique(df[col].dropna()))
        vals_map = {str(v).lower(): v for v in vals}

        for order in order_lists:
            if set(vals_map).issubset({o.lower() for o in order}):
                result[col] = [vals_map[o.lower()] for o in order if o.lower() in vals_map]
                break

    return result

class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X = pd.DataFrame(X).copy()
        expressions = EXPRESSIONS
        for expr in expressions:
            X = X.eval(expr, engine="python")
        bool_cols = X.select_dtypes(include="bool").columns
        X[bool_cols] = X[bool_cols].astype(int)

        return X

class CategoricalTyper(BaseEstimator, TransformerMixin):
    """
    Orders categorical values (i.e No < Yes, Low < Medium < High, etc) for graphs.
    Also, change the dtype of cat_cols into 'category'
    """

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.cat_cols_ = [
            col for col in X.columns
            if not is_num_check(col, df=X)
        ]
        self.ordered_cats_ = {
            col: (
                sorted(X[col].unique(), key=cat_order)
                if not pd.api.types.is_numeric_dtype(X[col])
                else sorted(X[col].unique())
            )
            for col in self.cat_cols_
        }
        return self

    def transform(self, X, y=None):
        X = pd.DataFrame(X).copy()
        for col, cats in self.ordered_cats_.items():
            if col in X.columns:
                X[col] = pd.Categorical(X[col], categories=cats, ordered=True)
        obj_cols = X.select_dtypes(include=["object", "string"]).columns
        X[obj_cols] = X[obj_cols].astype("category")
        return X

class OrdinalEncoderTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        non_target_cols = [c for c in X.columns if c not in TARGET]
        ordered = detect_ordinal_order_cols(X[non_target_cols])
        self.mappings_ = {
            col: {cat: i for i, cat in enumerate(order)}
            for col, order in ordered.items()
        }
        return self

    def transform(self, X, y=None):
        X = pd.DataFrame(X).copy()
        for col, mapping in self.mappings_.items():
            if col in X.columns:
                X[col] = X[col].map(mapping)
        return X


def split_num_cat(columns, for_graph=False, df=None):
    num_cols = []
    cat_cols = []

    for col in columns:
        if is_num_check(col, for_graph=for_graph, df=df):
            num_cols.append(col)
        else:
            cat_cols.append(col)

    return [num_cols, cat_cols]

pretrain_pipeline = Pipeline([
    ("feature_engineer", FeatureEngineer()),
    ("categorical_typer", CategoricalTyper()),
    ("ordinal_encoder",   OrdinalEncoderTransformer()),
])

TARGET_IS_NUM = is_num_check(TARGET)
HAS_NULLS = train_df.isna().any().any() or  test_df.isna().any().any()
if HAS_NULLS:
    raise ValueError('You need to implement imputer')




In [ ]:
if SPEED_UP:
    train_df = train_df[:1000].copy()
    test_df = test_df[:500].copy()
    sample_df = sample_df[:500].copy()

In [ ]:
if DO_IDA:
    categorical_typer = CategoricalTyper()
    plot_df = categorical_typer.fit_transform(train_df)

    ORDERED_CATS = categorical_typer.ordered_cats_

    COLUMNS = get_columns(df=plot_df)
    COLUMNS_X = get_predictor_names(df=plot_df)

    num_cols, cat_cols = split_num_cat(COLUMNS, df=plot_df, for_graph=True)
    num_cols_X, cat_cols_X = split_num_cat(COLUMNS_X, df=plot_df, for_graph=True)

def IDA(func):
    if DO_IDA: return func
    else:
        def empty_func(*args, **kwargs):
            pass
    return empty_func

@IDA
def plot_num_vs_cat(ax, x, y, df=None):
    if df is None:
        df = plot_df
    df.boxplot(column=y, by=x, ax=ax)

@IDA
def plot_num_vs_num(ax, x_col, y_col, df=None, n_bins=20):
    if df is None:
        df = plot_df
    x, y = df[x_col], df[y_col]

    bins = np.linspace(x.min(), x.max(), n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_means = [y[(x >= bins[i]) & (x < bins[i+1])].mean() for i in range(n_bins)]

    ax.scatter(x, y, alpha=0.3, s=15, label="data")
    ax.plot(bin_centers, bin_means, color="red", linewidth=2, marker="o", markersize=4, label="binned mean")
    ax.set(xlabel=x_col, ylabel=y_col)

@IDA
def plot_cat_vs_any(ax, x, y, df=None, mode='fill'):
    if df is None:
        df = plot_df
    hue_order = list(reversed(ORDERED_CATS[y]))
    match mode:
        case 'fill':
            multiple, stat = 'fill', 'proportion'
        case 'stack':
            multiple, stat = 'stack', 'count'
    sns.histplot(data=df, x=x, hue=y, hue_order=hue_order,
                 stat=stat, multiple=multiple, ax=ax)

@IDA
def plot(ax, x, y, df=None):
    if df is None:
        df = plot_df
    if y in cat_cols:
        plot_cat_vs_any(ax, x, y, df)
    elif x in num_cols:
        plot_num_vs_num(ax, x, y, df)
    else:
        plot_num_vs_cat(ax, x, y, df)

@IDA
def graphs(df=None,target_only=False):
    if df is None:
        df = plot_df
    if target_only:
        cols_X = COLUMNS_X
        cols_Y = TARGET
    else:
        cols_X = COLUMNS
        cols_Y = COLUMNS

    size =   (8, 8)
    fig, ax = plt.subplots(
          len(cols_X),
          len(cols_Y),
          squeeze=False,
          figsize=(size[0] * len(cols_Y), size[1] * len(cols_X))
          )


    for i, x in enumerate(cols_X):
        for j, y in enumerate(cols_Y):
            if i != j:
              plot(ax[i,j], x, y, df)

    plt.show()



In [ ]:
graphs()

In [ ]:
@IDA
def print_meta_data():
    print("Train shape: ", train_df.shape, "Test Shape: ", test_df.shape)
    print("\n\n"+"-"*40+"\n")

    print(train_df.describe().T)
    print("\n\n"+"-"*40+"\n")

    print("Nulls:\n\n", train_df.isnull().sum().sort_values())
    print("\n\n"+"-"*40+"\n")

    print("Uniques:\n\n",train_df.nunique().sort_values())
    print("\n\n"+"-"*40+"\n")

    train_df.head(10)

@IDA
def correlation_matrices():
    df = train_df[TARGET + list(COLUMNS_X)] #Make the target first
    df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
    sns.heatmap(df.corr(numeric_only=True), cmap="BrBG", annot=True)
    plt.show()

    fig, ax = plt.subplots(figsize=(20,20))
    sns.heatmap(df_encoded.corr(), cmap="BrBG", annot=True,ax=ax)
    plt.show()


In [ ]:
print_meta_data()


In [ ]:
correlation_matrices()

In [ ]:
if DO_IDA:
    del plot_df

train_df = pretrain_pipeline.fit_transform(train_df)
test_df  = pretrain_pipeline.transform(test_df)

ORDERED_CATS = pretrain_pipeline.named_steps["categorical_typer"].ordered_cats_

COLUMNS = get_columns()
COLUMNS_X = get_predictor_names()

num_cols, cat_cols = split_num_cat(COLUMNS)
num_cols_X, cat_cols_X = split_num_cat(COLUMNS_X)

def generate_target_transforms():
    """This rather long function generates two transforms. One is direct transform. It transforms categorical variables into ordinals and does nothing otherwise.
    It also generates inverse transform depends on the aim of the prediction. If it is probability it will return it. If it is category it will take argument max of prob.
    Note that direct and inverse transforms should be applied to the whole column.
    The basic idea is that direct transform is applied before training, and inverse transform is applied to probability prediction.
    This should be enough to generate correct format of the submission without any interference."""

    def f(y):
        return y.values
    def inv_f(y):
        return y
    if train_df[TARGET].shape[1] == 1:

        if TARGET_IS_NUM:
            pass

        else:
            if TARGET[0] in ORDERED_CATS:
                order = ORDERED_CATS[TARGET[0]]
            else:
                order = train_df[TARGET[0]].dropna().unique()
            order = np.array(order)
            dic = {cat: i for i, cat in enumerate(order)}
            def f(y):
                y = y.squeeze()
                return y.map(f.dic).astype(int).values
            f.dic = dic

            match PREDICTION_AIM:
                case 'probability':
                    if len(order) == 2:
                        def inv_f(p):
                            return p[:,1]
                    else:
                        def inv_f(p):
                            return p[:,1:]
                case 'category':
                    def inv_f(p):
                        return order[np.argmax(p, axis=1)]
    else:
        raise ValueError('Multicolumn target is not implemented yet.')
    return f, inv_f

TARGET_TRANSFORM, TARGET_INV_TRANSFORM = generate_target_transforms()
del generate_target_transforms

TARGET_SHAPE_WIDTH = 1 if TARGET_IS_NUM else len(train_df[TARGET[0]].dropna().unique())

In [ ]:
detect_ordinal_order_cols(train_df, order_lists=ORDER_LISTS)

{'Irrigation_Need': ['Low', 'Medium', 'High']}

In [ ]:
TARGET[0] in ORDERED_CATS

True

In [ ]:

def detect_ordinal_order_cols(df, order_lists=ORDER_LISTS):
    """
    Generates a dict with keys being columns that contain ordered categonical variables, and values are lists of unique values in natural order.
    """
    result = {}
    for col in df.columns:
        values = set(df[col].dropna().unique())
        for order in order_lists:
            if values.issubset(set(order)):
                result[col] = order
    return result


In [ ]:



# Ordinal encoding for ordered categories
# def split_ordered_unordered(cols, ord_cat_cols=ord_cat_cols):
#     ord, unord = [], []
#     for col in cols:
#         if col in ord_cat_cols:
#             ord.append(col)
#         else:
#             unord.append(col)
#     return ord, unord

# ord_cat_cols, unord_cat_cols = split_num_cat(cat_cols)
# ord_cat_cols_X, unord_cat_cols_X = split_num_cat(cat_cols_X)

In [ ]:
match SCORING:
    case 'roc_auc':
        from sklearn.metrics import roc_auc_score
        scoring = roc_auc_score
    case 'balanced_accuracy':
        from sklearn.metrics import balanced_accuracy_score
        def balanced_accuracy_score_prob(y_val, y_prob):
            y_pred = np.argmax(y_prob, axis=1)
            return balanced_accuracy_score(y_val, y_pred)
        scoring = balanced_accuracy_score_prob

seed_seq = np.random.SeedSequence(SEED)

def sample_parameters(dist, rng=None):
    """
    Function samples random distribution of parameters. It handle multiple threads by seed sequence.
    It is not reproducible if rng == None. If you want to control reproducability pass different fixed rngs to the different threads.
    """
    if rng is None:
        rng = np.random.default_rng(None)

    items = sorted(dist.items())
    params = dict()

    for k, v in items:
        if hasattr(v, "rvs"):
            params[k] = v.rvs(random_state=rng)
        elif isinstance(v, (list, tuple)):     # Be careful with generalization of this. It must give False for str.
            params[k] = v[rng.integers(len(v))]
        else:
            params[k] = v
    return params

class ModelRegistry:
    """Class that associates name of models with their Model cls and also creates a list of models appropriate for each task."""

    def __init__(self):
        self._models = {}
        self._model_lists = {}

    def add(self, name, Model_class, purposes, lower, upper):
        if not isinstance(purposes, (list, tuple)):
            purposes = [purposes]
        self._models[name] = Model_class
        for purpose in purposes:
            self._model_lists.setdefault(purpose, []).append((name, lower, upper))

    def get_purpose(self):
        if not TARGET_IS_NUM:
            if len(TARGET) == 1:
                return 'single_target_prob_pred'
        raise ValueError('Purpose generator cannot generate a purpose for the task at hand.')

    def get_list(self, purpose=None):
        purpose = purpose or self.get_purpose()
        return self._model_lists[purpose]

    def __getitem__(self, name):
        if name not in self._models:
            raise ValueError(f'Model {name} not found in registry.')
        return self._models[name]

registry = ModelRegistry()

def register_model(name, purposes, lower=5, upper=30):
    """
    Whenever you add new model, this decorator adds a name and purpose for it into the registry.
    Note that single model may be appropriate for different tasks. purposes may be a list of or a single str.
    lower - min number of this model type in the ladder.
    upper - same but max.

    Current purposes:
    'single_target_prob_pred' - for binary single column target probability prediction.
    """

    def decorator(cls):
        registry.add(name=name, Model_class=cls, purposes=purposes, lower=lower, upper=upper)
        cls._model_name = name
        return cls

    return decorator



class Model(ABC):
    """
    Abstract base class. Instances are supposed to implement specific models. The model is supposed to be the whole pipeline. Onle
    """

    vars_to_save = ['_param', '_oof'] #This will be saved in the file. Can be overriden in instances.
    _fit_params = {}

    def __init__(self, complexity=None, param_dist=None, rng=None):
        param_dist = param_dist or self.generate_distribution(complexity)
        self._param = sample_parameters(param_dist, rng=rng)
        self._oof = None
        self._model = self.build_pipeline(self._param)

    @abstractmethod
    def generate_distribution(self, complexity):
        pass

    @abstractmethod
    def build_pipeline(self, param):
        pass

    @property
    def name(self):
        return type(self)._model_name

    def fit(self, X, y):
        self._model.fit(X, y, **type(self)._fit_params)

    @property
    def params(self):
        return self._param

    def predict(self, X):
        if TARGET_IS_NUM:
            return self._model.predict(X)
        else:
            return self._model.predict_proba(X)

    @property
    def oof(self):
        return self._oof

    def set_oof(self, oof):
        self._oof = oof

    def save(self, filepath):
        name = (self.name, )
        values = tuple(getattr(self, var) for var in self.vars_to_save)
        with open(filepath, 'wb') as f:
            pickle.dump(name + values, f)

    @classmethod
    def load(cls, filepath):
        if not filepath.exists():
            raise ValueError('Loading a model with a missing data file.')
        with open(filepath, "rb") as f:
           name, *loaded_vars = pickle.load(f)

        model_cls = registry[name]
        model = model_cls(complexity=1.0) # complexity doesn't matter, because we change the parameters below.
        var_names = model_cls.vars_to_save
        for var_name, value in zip(var_names, loaded_vars):
            setattr(model, var_name, value)
        return model


class CrossValScore():
    """A class to do CV and store OOF results in the model"""

    def __init__(self, model, X, y, *, splits):
        self._scores = []
        y_oof = np.zeros((len(y), TARGET_SHAPE_WIDTH))
        for train_idx, val_idx in splits:
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            self._scores.append(scoring(y_val, y_pred))
            y_oof[val_idx] = y_pred
        self._scores = np.array(self._scores)
        if TARGET_SHAPE_WIDTH > 1:
            y_oof = y_oof[:, : -1]
        model.set_oof(y_oof)

    @property
    def score(self):
        return self._scores.mean(), self._scores.std()

@dataclass
class ModelEntry:
    score: float
    name: str
    file_path: str
    compute_time: int

    def load_model(self):
        return Model.load(self.file_path)

    def delete_file(self):
        if os.path.exists(self.file_path):
            os.remove(self.file_path)

    def __lt__(self, other):
        return self.score < other.score

@dataclass
class ModelClass:
    lower: int
    upper: int
    entries: list = field(default_factory=list)
    def __len__(self) -> int:
        return len(self.entries)

    def is_full(self) -> bool:
        return len(self) >= self.upper

    def is_satisfied(self, at_lower_bound_check=False) -> bool:
        # we check whether we can evict a model without breaking the bound len(self) >= self.lower
        if at_lower_bound_check:
            return len(self) > self.lower
        return len(self) >= self.lower

    def pop(self) -> None:
        worst = self.entries.pop(-1)
        worst.delete_file()

    def mean_score(self, top=10):
        if len(self) == 0: return None
        score, count = 0, 0
        for entry in self.entries[:top]:
            score += entry.score
            count += 1
        return score / count

    def insert(self, entry: ModelEntry) -> None:
        if self.is_full():
            if entry < self.entries[-1]:
                entry.delete_file()
                return
            self.pop()
        # Binary search insertion to keep list sorted descending
        lo, hi = 0, len(self.entries)
        while lo < hi:
            mid = (lo + hi) // 2
            if self.entries[mid].score > entry.score:
                lo = mid + 1
            else:
                hi = mid
        self.entries.insert(lo, entry)

class LeaderBoard:
    def __init__(self, num_models=100):
        self.classes = {}
        self._complexities = {}
        self.num_models = num_models
        os.makedirs(LEADERBOARD_STORAGE_DIR, exist_ok=True)

    def add_class(self, name, lower, upper):
        self.classes[name] = ModelClass(lower=lower, upper=upper)
        self._complexities[name] = 1.0

    def increase_complexity(self, name=None, val=0.5):
        if name is None:
            names = self._complexities.keys()
        else:
            names = [name]
        for name in names:
            self._complexities[name] += val
            self._complexities[name] = max(self._complexities[name], 1.0)

    def complexity(self, name):
        return self._complexities[name]

    def evaluate_models(self):
        """
        The function changes complexity based on the problem performance divided by log of computation time.
        We first subtract min score from all scores to make the positive. Then we divide each entry by log of compute time.
        Next, we subtract overall mean and divide by overall std to normalize the scores.
        The mean of this quanity/10 in each class is the change of complexity of that class.

        Note that value of complexities of different class are not related to each other.
        This function is an attempt to measure complexity by log of compute time.
        I don't think fairly judge the models.
         """
        cls_scores = {name: np.array([e.score for e in cl.entries]) for name, cl in self.classes.items()}
        times      = {name: np.array([e.compute_time for e in cl.entries]) for name, cl in self.classes.items()}

        min_score = min(s for scores in cls_scores.values() for s in scores)
        adjusted  = {name: (cls_scores[name] - min_score) / np.log1p(times[name])
                    for name in self.classes}

        all_adjusted = np.concatenate(list(adjusted.values()))
        mean_AS, std_AS = all_adjusted.mean(), all_adjusted.std()

        for name, adj in adjusted.items():
            final_score = ((adj - mean_AS + 0.25 * std_AS) / std_AS).mean()
            final_score /= 10
            self.increase_complexity(name=name, val=final_score)



    def __len__(self):
        return sum(len(c) for c in self.classes.values())

    def _pop(self, new_score):
        worst_score, candidate = new_score, None
        for name, cl in self.classes.items():
            if not cl.is_satisfied(at_lower_bound_check=True):
                continue

            worst_in_class = cl.entries[-1]
            if worst_in_class.score < worst_score:
                worst_score, candidate = worst_in_class.score, cl

        if candidate is None:
            return False

        candidate.pop()

        return True

    def add(self, class_name, model_entry):
        score = model_entry.score
        if len(self) >= self.num_models and not self._pop(score):
            model_entry.delete_file()
            return
        self.classes[class_name].insert(model_entry)

    def generate_model_entry(self, model, score, compute_time, class_name):
        now = datetime.now()
        model_name = class_name + now.strftime('_%Y%m%d_%H%M%S%f')[:-3]
        path = LEADERBOARD_STORAGE_DIR / model_name
        model.save(path)
        return class_name, ModelEntry(score=score, name=model_name, file_path=path, compute_time=compute_time)

    def __str__(self):
        table = []
        for name, cl in self.classes.items():
            for model in cl.entries:
                table.append((name, model.score))
        table.sort(key=lambda x: -x[1])
        output = f'{'Model':<20} | {'Score':>10}\n'
        output += "-" * 18 + '\n'
        for model, score in table:
            output +=f"{model:<30} | {score:>10.4f}" + '\n'
        output += f'Complexities of the models are {self._complexities}\n'
        return output

    def get(self):
        # Returns a next model name to try. Prioratizes models that don't saturate lower and better performing models
        models = []
        prob = []
        # We randomize the lookup over models, so it doesn't generate the first model with cl.is_satisfied()==False after multiple calls.
        rng = np.random.default_rng(seed_seq.spawn(1)[0])
        items = list(self.classes.items())
        rng.shuffle(items)
        for name, cl in items:
            if not cl.is_satisfied():
                return name
            models.append(name)
            prob.append(cl.mean_score())
        prob = np.array(prob, dtype=float)
        # Randomly select model
        if len(prob) == 0 or np.nanmax(prob) is None:
            return random.choice(list(self.classes.keys()))
        temp = prob.std() / 2
        prob = (prob - prob.mean()) / temp # We normalize the prob distribution to have std = 2
        prob = np.where(np.isnan(prob), np.nanmax(prob), prob)
        prob = np.exp(prob)
        prob = prob / prob.sum()
        return np.random.choice(models, p=prob)

    def get_best(self, length=20, min_repr=0):
        length = min(length, len(self))
        files = set()
        print('Picked models (minimal requirement):')
        if min_repr:
            for cl in self.classes.values():
                for entry in cl.entries[:min_repr]:
                    files.add((entry.name, entry.file_path))
                    print(f'Score: {entry.score}. Name: {entry.name}')
        print('Picked models (best score):')
        table = []
        for name, cl in self.classes.items():
            for entry in cl.entries:
                table.append((entry.score, (entry.name, entry.file_path)))

        table.sort(key=lambda x: -x[0])
        for score, data in table:
            if len(files) >= length:
                break
            print(f'Score: {score}. Name: {data[0]}')
            files.add(data)
        return list(files)

    def save(self):
        name = 'LeaderBoard'
        path = LEADERBOARD_STORAGE_DIR / name
        with open(path, "wb") as f:
            pickle.dump(self, f, protocol=pickle.HIGHEST_PROTOCOL)

    def load(self):
        name = 'LeaderBoard'
        path = LEADERBOARD_STORAGE_DIR / name
        if not path.exists():
            return False
        with open(path, "rb") as f:
            loaded = pickle.load(f)
        if isinstance(loaded, LeaderBoard):
            self.__dict__.update(loaded.__dict__)
            return True
        else:
            print(f'Loaded instance of the leader board was corrupted. Class of instance was {type(loaded)} instead of {type(self)}')
            return False


class Judge:
    def __init__(self, X, y, cv, X_test, num_models=100, model_list=None, ):
        self.X = X
        self.y = y
        self.X_test = X_test
        self.splits = list(cv.split(X, y))
        if model_list is None:
            model_list = registry.get_list()
        self.board=LeaderBoard(num_models=num_models)
        for name, lower, upper in model_list:
            self.board.add_class(name, lower, upper)

    def _evaluate_one(self, name, X, y, splits, rng=None):
        timer = time.perf_counter()

        model_cls = registry[name]
        model_complexity = self.board.complexity(name)
        model = model_cls(rng=rng, complexity=model_complexity)
        cv_results = CrossValScore(model, X, y, splits=splits)
        score, std = cv_results.score
        timer = time.perf_counter() - timer
        entry = self.board.generate_model_entry(model=model, score=score, compute_time=timer, class_name=name)
        timer = time.strftime("%H:%M:%S", time.gmtime(timer))
        msg = f'Tested new model in the class {name}. Score = {score:.4f} \u00b1 {std:.4f}. It took {timer}. \n Parameters are {model.params} \n'

        return entry + (msg,)

    def step(self, n_workers=-1):
        # Currently there are issues with the code here.
        # 1. The dataframe is being copied to each thread, but it is not altered.
        # So a better approach is to make them use a single df without copying.
        # The issue is that I don't know how to do it for a df.

        timer = time.perf_counter()
        STEP_BATCH_SIZE = 32
        # tmp = tempfile.mkdtemp()
        # X_path = os.path.join(tmp, 'X.pkl')
        # y_path = os.path.join(tmp, 'y.pkl')
        # splits_path = os.path.join(tmp, 'split.pkl')
        # np.save(X_path, self.X)
        # np.save(y_path, self.y)
        # np.save(splits_path, self.splits)

        names = [self.board.get() for _ in range(STEP_BATCH_SIZE)]
        rngs = [np.random.default_rng(s) for s in seed_seq.spawn(STEP_BATCH_SIZE)]

        # try:
        results = Parallel(n_jobs=n_workers, prefer="threads")(
            delayed(self._evaluate_one)(name, self.X, self.y, self.splits, rng=rng)
            for name, rng in zip(names, rngs)
        )
        # finally:
        #     os.remove(X_path)
        #     os.remove(y_path)
        #     os.remove(splits_path)

        for name, entry, msg in results:
            print(msg)
            self.board.add(name, entry)
        self.board.evaluate_models()
        timer = time.perf_counter() - timer
        time_spent = time.strftime("%H:%M:%S", time.gmtime(timer))
        print(self)
        print(f'Tested a batch of {STEP_BATCH_SIZE} model(s). It took {time_spent}.')
        return timer

    def construct_df(self, length=30, min_repr=1):
        ens_train_df = pd.DataFrame()
        ens_test_df = pd.DataFrame()
        file_paths = self.board.get_best(length=length, min_repr=min_repr)
        for name, path in file_paths:
            model = Model.load(path)
            model.fit(self.X, self.y)
            pred = model.predict(self.X_test)
            if model.oof.ndim > 1:
                for i in range(model.oof.shape[1]):
                    ens_train_df[name + f'{i}'] = model.oof[:,i]
                    ens_test_df[name + f'{i}'] =pred[:,i]
            else:
                ens_train_df[name] = model.oof
                ens_test_df[name] = pred


        return ens_train_df, ens_test_df

    def predict(self, model=None):
        if TARGET_IS_NUM:
            raise ValueError('Numerical target prediction is not realised.')
        else:
            match model:
                case None:
                    from sklearn.linear_model import LogisticRegression
                    model =  LogisticRegression()
                    param_dist = {
                        "random_state": [SEED],
                        "max_iter": [500],
                        "solver": ["lbfgs"],
                        "C": loguniform(1e-3, 1.0),
                       # "l1_ratio": uniform(0.05, 0.9),
                        "class_weight": ["balanced"],
                    }
                case _:
                    ValueError('Unknown prediction model')
        from sklearn.model_selection import RandomizedSearchCV
        search = RandomizedSearchCV(
            model,
            param_distributions=param_dist,
            n_iter=30,
            n_jobs=-1,
            cv=self.splits,
            scoring=SCORING
        )

        X_ens_train, X_ens_test = self.construct_df()
        search.fit(X_ens_train, self.y)

        best_score = search.best_score_
        best_index = search.best_index_
        best_std = search.cv_results_['std_test_score'][best_index]

        print(f'Ensembling is done. Best score: {best_score:.6f} \u00b1 {best_std:.6f}, Best params: {search.best_params_}')
        p_pred = search.best_estimator_.predict_proba(X_ens_test)
        return TARGET_INV_TRANSFORM(p_pred)

    def save(self):
       self.board.save()

    def load(self):
        if self.board.load():
            print(f'Loading of existing leaderboard is successful. The Leader board is: \n {self} \n\n')
        else:
            print('Loading failed. New leaderboard is created.\n')


    def __str__(self):
        return self.board.__str__()

In [ ]:


@register_model(name='LogisticRegression', purposes='single_target_prob_pred', lower=2, upper=5)
class LogisticRegressionModel(Model):
    def generate_distribution(self, complexity):
        k = complexity
        return {
            "model__random_state": SEED,
            "model__max_iter": int(400 * k),
            "model__C": loguniform(1e-2 * k, 1.0 * k),
          #  "model__l1_ratio": uniform(0.05, 0.9),
            "model__solver": "lbfgs",
            "model__class_weight": "balanced",
        }

    def build_pipeline(self, param):
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler, OneHotEncoder
        from sklearn.compose import ColumnTransformer

        num_prep = Pipeline(steps=[('scaler', StandardScaler())])
        cat_prep = Pipeline(steps=[
            ('ohe', OneHotEncoder(handle_unknown="error"))
        ])

        preprocessor = ColumnTransformer(transformers=[
            ("num", num_prep, num_cols_X),
            ("cat", cat_prep, cat_cols_X)
        ])

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', LogisticRegression()),
        ])
        pipe.set_params(**param)

        return pipe


@register_model(name='HistGBClassifier', purposes='single_target_prob_pred')
class HistGBClassifierModel(Model):
    def generate_distribution(self, complexity):
        k = complexity
        return {
            "model__early_stopping": True,
            "model__scoring": SCORING,
            "model__validation_fraction": 0.1,
            "model__n_iter_no_change": randint(int(10 * k), int(20 * k)),
            "model__random_state": SEED,
            "model__max_iter": randint(int(500 * k), int(1000 * k)),
            "model__learning_rate": uniform(0.001, 0.1 / k),
            "model__max_depth": randint(max(2, int(3 * k)), max(3, int(10 * k))),
            "model__max_leaf_nodes": randint(max(2, int(20 * k)), max(3, int(150 * k))),
            "model__min_samples_leaf": randint(1, max(2, int(50 / k))),
            "model__l2_regularization": uniform(0.0, max(1e-6, 1.0 / k)),
            "model__max_features": uniform(min(0.99, 0.5 + 0.5 * (1 - 1/k)), min(0.01, 0.5 / k)),
            "model__class_weight": ["balanced"],
        }

    def build_pipeline(self, param):
        from sklearn.ensemble import HistGradientBoostingClassifier
        from sklearn.compose import ColumnTransformer


        numerical_columns = num_cols_X
        categorical_columns = cat_cols_X

        cat_indices = list(range(len(numerical_columns), len(numerical_columns) + len(categorical_columns)))

        preprocessor = ColumnTransformer(transformers=[
            ("num", Pipeline([('passthrough', 'passthrough')]), numerical_columns),
            ("cat", Pipeline([('passthrough', 'passthrough')]), categorical_columns),
        ])

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', HistGradientBoostingClassifier(categorical_features=cat_indices)),
        ])
        pipe.set_params(**param)
        return pipe

@register_model(name='LGBMClassifier', purposes='single_target_prob_pred')
class LGBMClassifierModel(Model):
    _fit_params = {"model__categorical_feature": "auto"}
  #  _fit_params = {"model__categorical_feature": [f"cat__{c}" for c in unord_cat_cols_X]}

    def generate_distribution(self, complexity):
        k = complexity
        return {
            "model__verbose": -1,
            "model__random_state": SEED,
            "model__n_jobs": 1,
            "model__n_estimators": randint(int(50 * k), int(150 * k)),
            "model__learning_rate": uniform(0.001, max(0.001, 0.1 / k)),
            "model__max_depth": randint(max(2, int(3 * k)), max(3, int(10 * k))),
            "model__num_leaves": randint(max(2, int(20 * k)), max(3, int(150 * k))),
            "model__min_child_samples": randint(1, max(2, int(50 / k))),
            "model__subsample": uniform(0.5, 0.5),
            "model__subsample_freq": randint(1, 10),
            "model__colsample_bytree": uniform(0.5, 0.5),
            "model__reg_alpha": loguniform(max(1e-8, 1e-5 / k), max(1e-7, 1.0 / k)),
            "model__reg_lambda": uniform(0.0, max(0.01, 10.0 / k)),
            "model__class_weights": 'balanced',
        }


    def build_pipeline(self, param):
        from lightgbm import LGBMClassifier
        from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
        from sklearn.compose import ColumnTransformer

        numerical_columns = num_cols_X
        categorical_columns = cat_cols_X

        cat_indices = list(range(len(numerical_columns), len(numerical_columns) + len(categorical_columns)))

        cat_prep = Pipeline([
            ('ordinal', OrdinalEncoder()),
            ('to_cat', FunctionTransformer(lambda X: X.astype('category'))),
        ])

        preprocessor = ColumnTransformer(transformers=[
            ("num", Pipeline([('passthrough', 'passthrough')]), numerical_columns),
            ("cat", cat_prep, categorical_columns),
        ]).set_output(transform='pandas')

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', LGBMClassifier()),
        ])
        pipe.set_params(**param)
        return pipe

@register_model(name='CatBoostClassifier', purposes='single_target_prob_pred')
class CatBoostClassifierModel(Model):
    def generate_distribution(self, complexity):
        k = complexity
        return {
            'model__bootstrap_type': 'Bernoulli',
            "model__verbose": 0,
            "model__random_seed": SEED,
            "model__thread_count": 1,
            "model__iterations": randint(int(50 * k), int(150 * k)),
            "model__learning_rate": uniform(0.001, max(0.001, 0.1 / k)),
            "model__depth": randint(min(14,int(3 * k)), min(16, int(10 * k))),
            "model__min_data_in_leaf": randint(1, max(2, int(50 / k))),
            "model__subsample": uniform(0.5, 0.5),
            "model__colsample_bylevel": uniform(0.5, 0.5),
            "model__l2_leaf_reg": uniform(0.0, max(0.01, 10.0 / k)),
            "model__border_count": randint(
                min(32, int(32 * k)),
                min(255, int(255 * k)),
            ),
            "model__auto_class_weights": 'Balanced',
        }

    def build_pipeline(self, param):
        from catboost import CatBoostClassifier
        from sklearn.compose import ColumnTransformer

        numerical_columns = num_cols_X
        categorical_columns = cat_cols_X

        cat_indices = list(range(len(numerical_columns), len(numerical_columns) + len(categorical_columns)))

        preprocessor = ColumnTransformer(transformers=[
            ("num", Pipeline([('passthrough', 'passthrough')]), numerical_columns),
            ("cat", Pipeline([('passthrough', 'passthrough')]), categorical_columns),
        ])

        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('model', CatBoostClassifier(cat_features=cat_indices,
                                         allow_writing_files=False)),
        ])
        pipe.set_params(**param)
        return pipe

@register_model(name='XGBClassifier', purposes='single_target_prob_pred')
class XGBClassifierModel(Model):
    def generate_distribution(self, complexity):
        k = complexity
        return {
            "model__tree_method": 'hist',
            "model__eval_metric": 'auc',
            "model__enable_categorical": True,
            "model__verbosity": 0,
            "model__n_jobs": 1,
            "model__random_state": SEED,
            "model__n_estimators": randint(int(50 * k), int(150 * k)),
            "model__learning_rate": uniform(0.001, max(0.001, 0.1 / k)),
            "model__max_depth": randint(max(2, int(3 * k)), max(3, int(10 * k))),
            "model__min_child_weight": randint(1, max(2, int(10 / k))),
            "model__subsample": uniform(0.5, 0.5),
            "model__colsample_bytree": uniform(0.5, 0.5),
            "model__gamma": uniform(0.0, max(0.01, 1.0 / k)),
            "model__reg_alpha": loguniform(max(1e-8, 1e-5 / k), max(1e-7, 1.0 / k)),
            "model__reg_lambda": uniform(0.0, max(0.01, 10.0 / k)),
        }

    def build_pipeline(self, param):
        from xgboost import XGBClassifier
        from sklearn.compose import ColumnTransformer

        numerical_columns = num_cols_X
        categorical_columns = cat_cols_X

        preprocessor = ColumnTransformer(transformers=[
            ("num", Pipeline([('passthrough', 'passthrough')]), numerical_columns),
            ("cat", Pipeline([('passthrough', 'passthrough')]), categorical_columns),
        ]).set_output(transform='pandas')

        pipe = Pipeline([('preprocessor', preprocessor), ('model', XGBClassifier())])
        pipe.set_params(**param)
        return pipe



@register_model(name='RandomForestClassifier', purposes='single_target_prob_pred', lower=3, upper=10)
class RandomForestClassifierModel(Model):
    def generate_distribution(self, complexity):
        k = complexity
        return {
            "model__n_jobs": 1,
            "model__random_state": SEED,
            "model__n_estimators": randint(int(100 * k), int(200 * k)),
            "model__max_depth": randint(int(5 * k), int(20 * k)),
            "model__min_samples_split": randint(2, max(3, int(20 / k))),
            "model__min_samples_leaf": randint(1, max(2, int(5 / k))),
            "model__max_features": ["sqrt", "log2"],
            "model__class_weight": "balanced_subsample",
        }

    def build_pipeline(self, param):
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.preprocessing import OneHotEncoder
        from sklearn.compose import ColumnTransformer

        numerical_columns = num_cols_X
        categorical_columns = cat_cols_X

        preprocessor = ColumnTransformer(transformers=[
            ("num", Pipeline([('passthrough', 'passthrough')]), numerical_columns),
            ("cat", Pipeline([('ohe', OneHotEncoder(handle_unknown="error"))]), categorical_columns),
        ])
        pipe = Pipeline([('preprocessor', preprocessor), ('model', RandomForestClassifier())])
        pipe.set_params(**param)
        return pipe

In [ ]:
def trainer(n_steps=500, num_models=100):
    start_time = time.perf_counter()


    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X = train_df[COLUMNS_X]
    X_test = test_df[COLUMNS_X]
    y = TARGET_TRANSFORM(train_df[TARGET])
    judge = Judge(
        X=X,
        y=y,
        X_test=X_test,
        cv=cv,
        num_models=num_models
        )
    judge.load()

    for i in range(n_steps):
        compute_time = judge.step()
        judge.save()
        print(f'{i+1} steps done out of {n_steps}.\n')
        current_time = time.perf_counter() - start_time
        if current_time + 3 * compute_time > MAX_RUNNING_TIME:
            print('We are low on time and stop the training cycle.')
            break

    y_pred = judge.predict()

    return y_pred

def generate_submission(y):
    result = sample_df.copy()
    result[TARGET[0]] = y
    result = result[list(sample_df.columns)]
    result.to_csv(SUBMISSION_FILE_NAME + '.csv', index=False)
    print(result.head(10))



In [ ]:
y_pred = trainer(n_steps = N_STEPS)

Loading of existing leaderboard is successful. The Leader board is: 
 Model                |      Score
------------------
CatBoostClassifier             |     0.9710
CatBoostClassifier             |     0.9675
CatBoostClassifier             |     0.9670
CatBoostClassifier             |     0.9612
CatBoostClassifier             |     0.9587
CatBoostClassifier             |     0.9578
CatBoostClassifier             |     0.9575
CatBoostClassifier             |     0.9571
CatBoostClassifier             |     0.9552
CatBoostClassifier             |     0.9540
CatBoostClassifier             |     0.9445
CatBoostClassifier             |     0.9444
CatBoostClassifier             |     0.9407
CatBoostClassifier             |     0.9389
LogisticRegression             |     0.9379
CatBoostClassifier             |     0.9271
LogisticRegression             |     0.9246
LogisticRegression             |     0.9238
LogisticRegression             |     0.9229
LogisticRegression             |     0.92

In [ ]:
generate_submission(y_pred)

       id Irrigation_Need
0  630000             Low
1  630001          Medium
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007            High
8  630008            High
9  630009          Medium
